### ⚠️ Patched for Local Execution
This notebook was originally designed for Google Colab. It has been automatically patched:
- Google Colab-specific imports and `drive.mount()` calls have been commented out
- Colab file paths (`/content/drive/...`) have been replaced with relative paths (`./`)
- `!pip install` commands have been commented out (install packages in your venv instead)

**To run locally:** activate your Python virtual environment first, then run this notebook in VS Code or Jupyter.

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
# [PATCHED] from google.colab import drive
# [PATCHED] drive.mount('./')

In [ ]:
working_folder = './'

photos_folder=working_folder + 'photos/'

In [ ]:
classes_names = ['Anger', 'Disgust', 'Fear', 'Happiness', 'Sadness', 'Surprise', 'Neutral']

In [ ]:
model_id='google/vit-base-patch16-224-in21k'

In [ ]:
import torch
import torch.nn as nn
from transformers import ViTModel
from transformers import ViTImageProcessor
from transformers.modeling_outputs import SequenceClassifierOutput

In [ ]:
class ViTForImageClassification(nn.Module):

    def __init__(self, num_labels=7):

        super(ViTForImageClassification, self).__init__()

        self.vit = ViTModel.from_pretrained(model_id)

        self.dropout = nn.Dropout(0.1)

        self.classifier = nn.Linear(self.vit.config.hidden_size, num_labels)

        self.num_labels = num_labels

    def forward(self, pixel_values, labels=None):

        outputs = self.vit(pixel_values=pixel_values)

        output = self.dropout(outputs.last_hidden_state[:, 0])

        logits = self.classifier(output)
        if labels is not None:

          if isinstance(labels, list):
            labels = torch.tensor(labels, dtype=torch.long, device=logits.device)

          loss_fct = nn.CrossEntropyLoss()
          loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

          return SequenceClassifierOutput(
              loss=loss,
              logits=logits,
              hidden_states=outputs.hidden_states,
              attentions=outputs.attentions
          )
        else:
          return logits

In [ ]:
model = ViTForImageClassification()

model = torch.load(working_folder + 'ViT_fine_Tuned_FED', map_location=torch.device('cpu'))


In [ ]:
from PIL import Image

image_path = photos_folder+'Happiness7.png'

image = Image.open(image_path).convert('RGB')
image

In [ ]:
import numpy as np

image_np = np.array(image)

In [ ]:
processor = ViTImageProcessor.from_pretrained(model_id)

In [ ]:
inputs = processor(images=image_np, return_tensors="pt")

In [ ]:
logits = model(**inputs)
logits

In [ ]:
predicted_index = torch.argmax(logits, dim=1).item()

predicted_index

In [ ]:
classes_names[predicted_index]

In [ ]:
def classify_image(image_path):

  image = Image.open(image_path).convert('RGB')

  image_np = np.array(image)

  inputs = processor(images=image_np, return_tensors="pt")

  logits = model(**inputs)

  predicted_index = torch.argmax(logits, dim=1).item()

  return classes_names[predicted_index]

In [ ]:
image_path = photos_folder + 'Fear2.png'

classified = classify_image(image_path)
classified